# Extracción MongoDB — `reportes.log_comunicaciones`
**Filtro:** `cola_id = 43` | **Fecha:** hoy (fecha dinámica)

Conecta a MongoDB, extrae los registros del día actual, normaliza **todas las columnas** del schema (`prueba.json`) y muestra un análisis de nulos por columna.

In [8]:
# ── Dependencias ──────────────────────────────────────────────────────────────
# pip install pymongo pandas dnspython
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pymongo import MongoClient

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [9]:
# ── Configuración de conexión ─────────────────────────────────────────────────
host_mongo = "44.195.114.98"
user_mongo = "bi_guest"
pwd_mongo = "gu3$t202X#"
db_name_mongo = "reportes"

MONGO_URI        = f"mongodb://{user_mongo}:{pwd_mongo}@{host_mongo}:27017/{db_name_mongo}"
DATABASE         = "reportes"
COLLECTION       = "log_comunicaciones"
COLA_ID          = 43
SCHEMA_JSON_PATH = Path(r"C:\Users\USUARIO\Downloads\NTP_CODEX\prueba.json")

In [10]:
# ── Rango del día de hoy (UTC) ────────────────────────────────────────────────
hoy       = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
manana    = hoy.replace(hour=23, minute=59, second=59, microsecond=999999)
print(f"Extrayendo registros del: {hoy.strftime('%Y-%m-%d')}")
print(f"  Desde : {hoy}")
print(f"  Hasta : {manana}")

Extrayendo registros del: 2026-04-09
  Desde : 2026-04-09 00:00:00+00:00
  Hasta : 2026-04-09 23:59:59.999999+00:00


In [11]:
# ── Leer columnas esperadas desde prueba.json ─────────────────────────────────
with open(SCHEMA_JSON_PATH, encoding='utf-8') as f:
    schema_raw = json.load(f)

props = schema_raw["collections"]["reportes.log_comunicaciones"]["jsonSchema"]["properties"]

# Columnas raíz (excluye los objetos anidados que se expanden aparte)
NESTED_KEYS = {"contacto", "detalle", "gestiones", "gestiones_old", "marcador_info"}

root_cols = [k for k in props.keys() if k not in NESTED_KEYS]

# Sub-columnas de contacto
contacto_cols = [
    f"contacto.{k}" for k in props["contacto"]["properties"].keys()
    if k != "_id"
]
contacto_cols_id = ["contacto._id.date", "contacto._id.timestamp"]

# Sub-columnas de marcador_info
marcador_cols = [
    f"marcador_info.{k}" for k in props["marcador_info"]["properties"].keys()
]

ALL_FLAT_COLS = root_cols + contacto_cols_id + contacto_cols + marcador_cols

print(f"Total columnas esperadas (flat): {len(ALL_FLAT_COLS)}")
print(f"  Raíz       : {len(root_cols)}")
print(f"  contacto   : {len(contacto_cols) + len(contacto_cols_id)}")
print(f"  marcador   : {len(marcador_cols)}")
print("\n(detalle[], gestiones[], gestiones_old[] quedan como listas — se analizan aparte)")

Total columnas esperadas (flat): 160
  Raíz       : 95
  contacto   : 30
  marcador   : 35

(detalle[], gestiones[], gestiones_old[] quedan como listas — se analizan aparte)


In [12]:
# ── Conexión y extracción ─────────────────────────────────────────────────────
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

# Verificar conexión
try:
    client.admin.command('ping')
    print("Conexión exitosa a MongoDB")
except Exception as e:
    print(f"Error de conexión: {e}")
    raise

db         = client[DATABASE]
collection = db[COLLECTION]

filtro = {
    "cola_id": COLA_ID,
    "fecha_ingreso_cola": {
        "$gte": hoy,
        "$lte": manana
    }
}

cursor   = collection.find(filtro)
registros = list(cursor)
print(f"Registros encontrados: {len(registros)}")

Conexión exitosa a MongoDB
Registros encontrados: 0


In [13]:
# ── Función de normalización flat ─────────────────────────────────────────────
def flatten_doc(doc: dict) -> dict:
    """Aplana un documento MongoDB al esquema definido en prueba.json."""
    flat = {}

    # Campos raíz
    for col in root_cols:
        flat[col] = doc.get(col, None)

    # contacto._id (objeto especial)
    contacto = doc.get("contacto", {})
    cid = contacto.get("_id", {})
    flat["contacto._id.date"]      = cid.get("date", None)      if isinstance(cid, dict) else None
    flat["contacto._id.timestamp"] = cid.get("timestamp", None) if isinstance(cid, dict) else None

    # contacto (resto de campos)
    for col in contacto_cols:
        key = col.split("contacto.", 1)[1]
        flat[col] = contacto.get(key, None)

    # marcador_info
    marcador = doc.get("marcador_info", {})
    for col in marcador_cols:
        key = col.split("marcador_info.", 1)[1]
        flat[col] = marcador.get(key, None) if isinstance(marcador, dict) else None

    # Arrays — se conservan como listas para análisis separado
    flat["detalle"]        = doc.get("detalle", [])
    flat["gestiones"]      = doc.get("gestiones", [])
    flat["gestiones_old"]  = doc.get("gestiones_old", [])

    return flat


# Normalizar todos los documentos
if registros:
    df = pd.DataFrame([flatten_doc(doc) for doc in registros])

    # Asegurar que todas las columnas del schema existen (aunque vengan vacías)
    for col in ALL_FLAT_COLS:
        if col not in df.columns:
            df[col] = None

    # Reordenar: columnas flat primero, luego arrays
    array_cols  = ["detalle", "gestiones", "gestiones_old"]
    ordered     = ALL_FLAT_COLS + array_cols
    extra_cols  = [c for c in df.columns if c not in ordered]
    df          = df[ordered + extra_cols]

    print(f"DataFrame shape: {df.shape}")
    print(f"Columnas totales: {len(df.columns)}")
else:
    df = pd.DataFrame(columns=ALL_FLAT_COLS + ["detalle", "gestiones", "gestiones_old"])
    print("Sin registros para hoy. Se crea DataFrame vacío con el schema completo.")

Sin registros para hoy. Se crea DataFrame vacío con el schema completo.


In [14]:
# ── Vista previa del DataFrame ────────────────────────────────────────────────
df.head(5)

,_id,agente_id,agente_nombre,alias_contacto,anio_fin_atencion,anio_ingreso,anio_inicio_atencion,arranque_id,callid,campana_id,campana_nombre,chan_id,codigo_atencion,cola_id,cola_nombre,comunicacion_id,correlativo,dia_semana_fin_atencion_id,dia_semana_fin_atencion_nombre,dia_semana_ingreso_id,dia_semana_ingreso_nombre,dia_semana_inicio_atencion_id,dia_semana_inicio_atencion_nombre,estado_atencion,etiqueta_id,etiqueta_nombre,extension,fecha_fin,fecha_fin_atencion,fecha_fin_atencion_index,fecha_fin_comunicacion,fecha_ingreso_cola,fecha_ingreso_cola_index,fecha_inicio,fecha_inicio_atencion,fecha_inicio_atencion_data_comparacion,fecha_inicio_atencion_index,fecha_inicio_comunicacion,fecha_inicio_comunicacion_index,fecha_primer_registro,fecha_primera_respuesta_agente,fin_timbrado_marcador,finalizada_por,gestion_unica,hora_fin_atencion_index,hora_ingreso_cola_index,hora_inicio_atencion_index,hora_inicio_comunicacion_index,identificacion,inicio_timbrado_marcador,instancia,isMarcador,lote_id,lote_nombre,mes_fin_atencion,mes_ingreso,mes_inicio_atencion,nodo_id,nodo_nombre,nombre_sub_resultado,origen,proveedor_id,proveedor_nombre,publicidad_id,publicidad_identificador,publicidad_nombre,realizado_tipificacion_obligatoria,resultado_id,resultado_marcador_id,resultado_marcador_nombre,servicio_id,servicio_secundario,sip_code,sub_resultado_id,sub_tipo_marcador_id,sub_tipo_marcador_nombre,supera_umbral,tiempo,tiempo_atencion,tiempo_cola,tiempo_conversacion,tiempo_gestion,tiempo_ring,tiempo_ring_acumulado,tiempo_ring_agente,tiempo_timbrado_marcador,tipo_comunicacion_id,tipo_marcador_id,tipo_marcador_nombre,tipo_telefono_marcador,whatsapp_id,workflow_nombre,workflow_numero,field-1,field-2,contacto._id.date,contacto._id.timestamp,contacto.campana,contacto.carga_id,contacto.chan_id,contacto.contacto_id,contacto.contacto_nombre,contacto.contacto_validado,contacto.correos,contacto.departamento,contacto.dirección,contacto.distrito,contacto.facebooks,contacto.fecha_creacion,contacto.fecha_modificacion,contacto.from_marcador,contacto.id_campana,contacto.id_contacto,contacto.isDirty,contacto.lote_id,contacto.nombre,contacto.nombre_base,contacto.nro_de_documento,contacto.numeros,contacto.otros_distritos,contacto.provincia,contacto.telefonos,contacto.tipo_campaña,contacto.tipo_de_campaña,contacto.twitters,marcador_info.arranque_id,marcador_info.chan_id,marcador_info.code,marcador_info.dato1,marcador_info.dato10,marcador_info.dato11,marcador_info.dato12,marcador_info.dato13,marcador_info.dato14,marcador_info.dato15,marcador_info.dato16,marcador_info.dato17,marcador_info.dato18,marcador_info.dato19,marcador_info.dato2,marcador_info.dato20,marcador_info.dato3,marcador_info.dato4,marcador_info.dato5,marcador_info.dato6,marcador_info.dato7,marcador_info.dato8,marcador_info.dato9,marcador_info.lote_id,marcador_info.lote_nombre,marcador_info.nombre,marcador_info.numero,marcador_info.register_id,marcador_info.resultado_id,marcador_info.sub_tipo_marcador_id,marcador_info.sub_tipo_marcador_nombre,marcador_info.tipo_marcador_id,marcador_info.tipo_marcador_nombre,marcador_info.tipo_numero,marcador_info.field-1,detalle,gestiones,gestiones_old


In [15]:
# ── Análisis de nulos por columna ─────────────────────────────────────────────
total = len(df)

nulos = pd.DataFrame({
    "columna"      : df.columns,
    "tipo_schema"  : [
        props.get(c.split(".")[0], {}).get("bsonType", "nested/array")
        if "." not in c else "nested"
        for c in df.columns
    ],
    "con_dato"     : [df[c].apply(lambda x: x is not None and not (isinstance(x, float) and pd.isna(x)) and x != [] and x != {}).sum() for c in df.columns],
    "nulos"        : df.isnull().sum().values,
    "vacios_lista" : [df[c].apply(lambda x: isinstance(x, list) and len(x) == 0).sum() for c in df.columns],
})

if total > 0:
    nulos["%_nulo"]     = (nulos["nulos"]        / total * 100).round(1)
    nulos["%_con_dato"] = (nulos["con_dato"]     / total * 100).round(1)
else:
    nulos["%_nulo"]     = 0.0
    nulos["%_con_dato"] = 0.0

nulos = nulos.sort_values("%_nulo", ascending=True).reset_index(drop=True)

print(f"Total registros analizados: {total}")
nulos

Total registros analizados: 0


,columna,tipo_schema,con_dato,nulos,vacios_lista,%_nulo,%_con_dato
0,_id,objectId,0,0,0,0.00,0.00
1,agente_id,int,0,0,0,0.00,0.00
2,agente_nombre,string,0,0,0,0.00,0.00
3,alias_contacto,string,0,0,0,0.00,0.00
4,anio_fin_atencion,int,0,0,0,0.00,0.00
...,...,...,...,...,...,...,...
158,marcador_info.tipo_numero,nested,0,0,0,0.00,0.00
159,marcador_info.field-1,nested,0,0,0,0.00,0.00
160,detalle,array,0,0,0,0.00,0.00
161,gestiones,array,0,0,0,0.00,0.00


In [16]:
# ── Columnas con datos (< 50% nulos) ─────────────────────────────────────────
cols_utiles = nulos[nulos["%_nulo"] < 50]["columna"].tolist()
print(f"Columnas con datos suficientes (< 50% nulos): {len(cols_utiles)}")
print(cols_utiles)

Columnas con datos suficientes (< 50% nulos): 163
['_id', 'agente_id', 'agente_nombre', 'alias_contacto', 'anio_fin_atencion', 'anio_ingreso', 'anio_inicio_atencion', 'arranque_id', 'callid', 'campana_id', 'campana_nombre', 'chan_id', 'codigo_atencion', 'cola_id', 'cola_nombre', 'comunicacion_id', 'correlativo', 'dia_semana_fin_atencion_id', 'dia_semana_fin_atencion_nombre', 'dia_semana_ingreso_id', 'dia_semana_ingreso_nombre', 'dia_semana_inicio_atencion_id', 'dia_semana_inicio_atencion_nombre', 'estado_atencion', 'etiqueta_id', 'etiqueta_nombre', 'extension', 'fecha_fin', 'fecha_fin_atencion', 'fecha_fin_atencion_index', 'fecha_fin_comunicacion', 'fecha_ingreso_cola', 'fecha_ingreso_cola_index', 'fecha_inicio', 'fecha_inicio_atencion', 'fecha_inicio_atencion_data_comparacion', 'fecha_inicio_atencion_index', 'fecha_inicio_comunicacion', 'fecha_inicio_comunicacion_index', 'fecha_primer_registro', 'fecha_primera_respuesta_agente', 'fin_timbrado_marcador', 'finalizada_por', 'gestion_unic

In [17]:
# ── Columnas completamente vacías ─────────────────────────────────────────────
cols_vacias = nulos[nulos["%_nulo"] == 100]["columna"].tolist()
print(f"Columnas 100% nulas (sin datos): {len(cols_vacias)}")
print(cols_vacias)

Columnas 100% nulas (sin datos): 0
[]


In [18]:
# ── Análisis de arrays: gestiones ─────────────────────────────────────────────
if total > 0 and df["gestiones"].apply(lambda x: isinstance(x, list) and len(x) > 0).any():
    gestiones_rows = []
    for idx, row in df.iterrows():
        comunicacion_id = row.get("comunicacion_id", idx)
        for g in (row["gestiones"] or []):
            g_flat = {"_doc_comunicacion_id": comunicacion_id}
            g_flat.update(g)
            gestiones_rows.append(g_flat)

    df_gestiones = pd.DataFrame(gestiones_rows)
    print(f"Total gestiones expandidas: {len(df_gestiones)}")
    display(df_gestiones.head(5))
else:
    print("Sin gestiones registradas hoy.")

Sin gestiones registradas hoy.


In [19]:
# ── Resumen ejecutivo ─────────────────────────────────────────────────────────
print("=" * 55)
print(f"  RESUMEN — cola_id={COLA_ID} | {hoy.strftime('%Y-%m-%d')}")
print("=" * 55)
print(f"  Registros extraídos       : {total}")
print(f"  Columnas en schema        : {len(ALL_FLAT_COLS)}")
print(f"  Columnas con datos        : {len(cols_utiles)}")
print(f"  Columnas 100%% vacías      : {len(cols_vacias)}")
if total > 0:
    print(f"  Agentes únicos            : {df['agente_id'].nunique()}")
    print(f"  Campañas únicas           : {df['campana_nombre'].nunique()}")
    print(f"  Tipos comunicación únicos : {df['tipo_comunicacion_id'].nunique()}")
print("=" * 55)

  RESUMEN — cola_id=43 | 2026-04-09
  Registros extraídos       : 0
  Columnas en schema        : 160
  Columnas con datos        : 163
  Columnas 100%% vacías      : 0
